# EvoSkill lab

> **SealQA question:** How many airlines are members of IATA, the organization that supports airline activity and helps formulate industry policy and standards?


**Terminology.** Two *induction* episodes create recorded failures. A failure analyzer reads them through shipped `PullAccess`; a separate builder emits one Markdown document inside shipped `meta.SkillSet`. Two untouched rows are *held out*. Inspired by [EvoSkill](https://www.sentient.xyz/blog/evoskill-automated-skill-induction-from-agent-failures).

## What changes

Only the skill augmentation changes. The baseline instructions, model, cases, evaluator, and bounds remain fixed. The path is 2 induction sessions, 1 failure analysis, 1 skill build, and 2 × 2 held-out comparisons: **8 sessions**.

## Live declaration

Provider: Claude. Pinned model: `claude-haiku-4-5-20251001`. Prerequisites: `uv sync --extra claude --group dev`, authenticate Claude Code, then explicitly set `META_EVOLVE_LIVE_CLAUDE=1`. Every inner session declares `MAX_TURNS = PULL_MAX_TURNS = 50`, `TIMEOUT_SECONDS = 1800`, and `INNER_BUDGET.wall_seconds = 1860`. These calls may incur cost.

In [ ]:
from pathlib import Path
import os, sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
lab = root / "examples" / "18_sealqa_research_labs"
if str(lab) not in sys.path:
    sys.path.insert(0, str(lab))

from sealqa_labs import ClaudeSessionRunner
from sealqa_labs.config import *

print({
    "provider": "Claude", "model": MODEL, "sessions": TOTAL_SESSIONS,
    "turns": MAX_TURNS, "timeout_seconds": TIMEOUT_SECONDS,
    "inner_wall_seconds": INNER_BUDGET.wall_seconds,
    "opt_in": f"{LIVE_OPT_IN}=1",
})


In [ ]:
if os.getenv(LIVE_OPT_IN) != "1":
    raise RuntimeError(
        f"Live-only lab: set {LIVE_OPT_IN}=1 after authenticating Claude Code."
    )

from sealqa_labs.evoskill import run

result = run(ClaudeSessionRunner())
print("sessions executed:", len(result.sessions))
print("usage:", result.usage)


## Visible record

The complete recorded induction failures are seed evaluation evidence. Pull-operation counters distinguish offered from used experience. The Markdown skill, procedures, checks, held-out ties/regressions/failures, lineage, and usage remain visible.

In [ ]:
for session in result.sessions:
    print(session.label, "failure=" + (session.failure.kind if session.failure else "none"))
print("artifacts:", result.artifacts)
print("measurements:", result.measurements)


## Live observation

No credentialed output is committed yet because the explicit opt-in was absent. Preserve the first complete result and state when the analyzer never read the offered failures; unused experience is a result, not a silent fallback.

## Audit and exercise

- Are rows 7–8 absent from induction and skill building?
- Do failure records omit evaluator-only answers and source URLs?
- Are baseline and augmented held-out sessions otherwise compatible?

**Change and predict:** remove verification checks from the skill-builder instruction. Predict held-out ties, regressions, and pull usage before rerunning.